# Unit 5 — Unity ML-Agents (Google Colab)

Two submissions: SnowballTarget and Pyramids. ML-Agents 1.1.0 requires **Python 3.10.12**, so this notebook uses a separate micromamba environment. Select a high-RAM Colab runtime if available.

**Run all cells from top to bottom.** Training and Hub uploads require network access and your Hugging Face write token.


## Install Unity ML-Agents and dependencies


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg libgl1 libglib2.0-0 libgomp1 libxcursor1 libxi6 libxrandr2 libxinerama1 >/dev/null
%pip -q install "huggingface_hub<1"
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
!./bin/micromamba create -y -p /content/hf-rl-unity -c conda-forge python=3.10.12 pip
!/content/hf-rl-unity/bin/python -m pip install -q "torch==2.2.2" "numpy<2" "mlagents==1.1.0" "huggingface_hub==0.25.2"
!/content/hf-rl-unity/bin/python -c "import sys, mlagents; print(sys.version, mlagents.__version__)"

## Download both Unity games and training configurations


In [ ]:
!mkdir -p config training-envs-executables/linux
!curl -fLsS https://raw.githubusercontent.com/panuza1/deep-rl-class/main/notebooks/unit5/config/SnowballTarget.yaml -o config/SnowballTarget.yaml
!curl -fLsS https://raw.githubusercontent.com/panuza1/deep-rl-class/main/notebooks/unit5/config/PyramidsRND.yaml -o config/PyramidsRND.yaml
!curl -fL "https://github.com/huggingface/Snowball-Target/raw/main/SnowballTarget.zip" -o training-envs-executables/linux/SnowballTarget.zip
!unzip -q -o training-envs-executables/linux/SnowballTarget.zip -d training-envs-executables/linux/
!chmod +x training-envs-executables/linux/SnowballTarget/SnowballTarget
!curl -fL "https://huggingface.co/spaces/unity/ML-Agents-Pyramids/resolve/main/Pyramids.zip" -o training-envs-executables/linux/Pyramids.zip
!unzip -q -o training-envs-executables/linux/Pyramids.zip -d training-envs-executables/linux/
!chmod +x training-envs-executables/linux/Pyramids/Pyramids

## SnowballTarget — train, then push to Hub


In [ ]:
!/content/hf-rl-unity/bin/mlagents-learn config/SnowballTarget.yaml --env=training-envs-executables/linux/SnowballTarget/SnowballTarget --run-id=SnowballTarget1 --no-graphics

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # Use a Hugging Face token with Write permission.

In [ ]:
!/content/hf-rl-unity/bin/mlagents-push-to-hf --run-id=SnowballTarget1 --local-dir=results/SnowballTarget1 --repo-id=panuza007/ppo-SnowballTarget --commit-message="Train SnowballTarget"

## Pyramids — train, then push to Hub

This Unity task may take several hours; Colab sessions can disconnect before training finishes.


In [ ]:
!/content/hf-rl-unity/bin/mlagents-learn config/PyramidsRND.yaml --env=training-envs-executables/linux/Pyramids/Pyramids --run-id=Pyramids1 --no-graphics

In [ ]:
!/content/hf-rl-unity/bin/mlagents-push-to-hf --run-id=Pyramids1 --local-dir=results/Pyramids1 --repo-id=panuza007/ppo-Pyramids --commit-message="Train Pyramids"